# Proyecto 1: Clasificador Completo de Cáncer de Mama

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 120 minutos  
**Conceptos integrados:** Regresión Logística, Árboles de Decisión, Random Forests, XGBoost, Tuning, Cross-Validation

## 🎯 Objetivo

Construir un **pipeline de clasificación end-to-end** que prediga si un tumor mamario es maligno o benigno usando el dataset Breast Cancer Wisconsin.

### Tareas Principales

1. ✅ Exploración y visualización de datos
2. ✅ Preparación y feature engineering
3. ✅ Comparación de múltiples modelos
4. ✅ Tuning de hiperparámetros
5. ✅ Evaluación exhaustiva con múltiples métricas
6. ✅ Análisis de importancia de features
7. ✅ Cross-validation y validación robusta
8. ✅ Selección del mejor modelo
9. ✅ Interpretación final y conclusiones

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Métricas
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc, confusion_matrix, 
    classification_report, precision_recall_curve
)

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
np.random.seed(42)

print("✅ Librerías importadas correctamente")

---

## 📊 1. Exploración y Análisis de Datos

In [ ]:
# Cargar dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

# Crear dataframe para análisis
df = pd.DataFrame(X, columns=cancer.feature_names)
df['target'] = y

print("🔬 Dataset: Breast Cancer Wisconsin")
print(f"\n📐 Dimensiones:")
print(f"   Total ejemplos: {X.shape[0]}")
print(f"   Total features: {X.shape[1]}")
print(f"\n📋 Características (primeras 5 features):")
for i, name in enumerate(cancer.feature_names[:5]):
    print(f"   {i+1}. {name}")
print(f"   ...\n   {len(cancer.feature_names)}. {cancer.feature_names[-1]}")

print(f"\n🎯 Distribución de Clases:")
print(f"   Clase 0 (Maligno): {np.sum(y == 0)} ejemplos ({np.sum(y == 0)/len(y)*100:.1f}%)")
print(f"   Clase 1 (Benigno): {np.sum(y == 1)} ejemplos ({np.sum(y == 1)/len(y)*100:.1f}%)")

In [ ]:
# Estadísticas descriptivas
print("\n📊 Estadísticas Descriptivas (primeras 5 features):\n")
print(df[cancer.feature_names[:5]].describe().round(2))

In [ ]:
# Visualización 1: Distribución de clases
fig = go.Figure()

classes = ['Maligno (0)', 'Benigno (1)']
counts = [np.sum(y == 0), np.sum(y == 1)]
colors = ['#FF6B6B', '#4ECDC4']

fig.add_trace(go.Bar(
    x=classes,
    y=counts,
    text=counts,
    textposition='outside',
    marker=dict(color=colors)
))

fig.update_layout(
    title="Distribución de Clases",
    xaxis_title="Clase",
    yaxis_title="Número de Ejemplos",
    template="plotly_white",
    height=400
)

fig.show()

print("\n💡 Dataset está razonablemente balanceado (212 maligno, 357 benigno)")

In [ ]:
# Visualización 2: Distribución de features importantes
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("mean radius", "mean concavity", "worst texture", "worst symmetry")
)

features_to_plot = ['mean radius', 'mean concavity', 'worst texture', 'worst symmetry']
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

for feature, (row, col) in zip(features_to_plot, positions):
    malignant = df[df['target'] == 0][feature]
    benign = df[df['target'] == 1][feature]
    
    fig.add_trace(
        go.Histogram(x=malignant, name='Maligno', opacity=0.7, marker_color='#FF6B6B', nbinsx=20),
        row=row, col=col
    )
    fig.add_trace(
        go.Histogram(x=benign, name='Benigno', opacity=0.7, marker_color='#4ECDC4', nbinsx=20),
        row=row, col=col
    )

fig.update_xaxes(title_text="Valor", row=2, col=1)
fig.update_xaxes(title_text="Valor", row=2, col=2)
fig.update_yaxes(title_text="Frecuencia", row=1, col=1)
fig.update_yaxes(title_text="Frecuencia", row=2, col=1)

fig.update_layout(
    title="Distribución de Features por Clase",
    template="plotly_white",
    height=600,
    hovermode='x unified'
)

fig.show()

In [ ]:
# Matriz de correlación
print("\n📈 Análisis de Correlación\n")

# Seleccionar top features por correlación con target
correlations = df.corr()['target'].drop('target').sort_values(ascending=False)

print("Top 10 Features más correlacionadas con el target:")
print(correlations.head(10).round(3))

print("\nTop 10 Features menos correlacionadas:")
print(correlations.tail(10).round(3))

---

## 🔧 2. Preparación de Datos

In [ ]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("✅ Train/Test Split:")
print(f"   Training set: {X_train.shape[0]} ejemplos")
print(f"   Test set: {X_test.shape[0]} ejemplos")
print(f"\n   Train - Clase 0: {np.sum(y_train == 0)} ({np.sum(y_train == 0)/len(y_train)*100:.1f}%)")
print(f"   Train - Clase 1: {np.sum(y_train == 1)} ({np.sum(y_train == 1)/len(y_train)*100:.1f}%)")
print(f"\n   Test - Clase 0: {np.sum(y_test == 0)} ({np.sum(y_test == 0)/len(y_test)*100:.1f}%)")
print(f"   Test - Clase 1: {np.sum(y_test == 1)} ({np.sum(y_test == 1)/len(y_test)*100:.1f}%)")

In [ ]:
# Normalización
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Normalización con StandardScaler:")
print(f"\n   Media (Train): {X_train_scaled.mean():.6f}")
print(f"   Desv. Est. (Train): {X_train_scaled.std():.6f}")
print(f"\n   Media (Test): {X_test_scaled.mean():.6f}")
print(f"   Desv. Est. (Test): {X_test_scaled.std():.6f}")

---

## 🤖 3. Comparación de Modelos Base

In [ ]:
# Definir modelos base
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, verbosity=0, n_jobs=-1)
}

# Entrenar y evaluar modelos base
results_base = {}

print("🏃 Entrenando modelos base...\n")

for name, model in models.items():
    # Entrenar
    model.fit(X_train_scaled, y_train)
    
    # Predicciones
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test = model.predict(X_test_scaled)
    
    if hasattr(model, 'predict_proba'):
        y_proba_train = model.predict_proba(X_train_scaled)[:, 1]
        y_proba_test = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba_train = model.decision_function(X_train_scaled)
        y_proba_test = model.decision_function(X_test_scaled)
        # Normalizar a [0, 1]
        y_proba_train = (y_proba_train - y_proba_train.min()) / (y_proba_train.max() - y_proba_train.min())
        y_proba_test = (y_proba_test - y_proba_test.min()) / (y_proba_test.max() - y_proba_test.min())
    
    # Métricas
    results_base[name] = {
        'model': model,
        'train_acc': accuracy_score(y_train, y_pred_train),
        'test_acc': accuracy_score(y_test, y_pred_test),
        'train_f1': f1_score(y_train, y_pred_train),
        'test_f1': f1_score(y_test, y_pred_test),
        'train_auc': roc_auc_score(y_train, y_proba_train),
        'test_auc': roc_auc_score(y_test, y_proba_test),
        'y_pred_test': y_pred_test,
        'y_proba_test': y_proba_test
    }
    
    print(f"✅ {name}")

In [ ]:
# Tabla resumen
summary_df = pd.DataFrame([
    {
        'Modelo': name,
        'Train Acc': f"{results_base[name]['train_acc']:.4f}",
        'Test Acc': f"{results_base[name]['test_acc']:.4f}",
        'Train F1': f"{results_base[name]['train_f1']:.4f}",
        'Test F1': f"{results_base[name]['test_f1']:.4f}",
        'Train AUC': f"{results_base[name]['train_auc']:.4f}",
        'Test AUC': f"{results_base[name]['test_auc']:.4f}",
    }
    for name in results_base.keys()
])

print("\n📊 Resultados de Modelos Base:\n")
print(summary_df.to_string(index=False))

In [ ]:
# Visualización comparativa
metrics = ['Test Acc', 'Test F1', 'Test AUC']
models_names = list(results_base.keys())
test_accs = [results_base[m]['test_acc'] for m in models_names]
test_f1s = [results_base[m]['test_f1'] for m in models_names]
test_aucs = [results_base[m]['test_auc'] for m in models_names]

fig = go.Figure()

fig.add_trace(go.Bar(name='Accuracy', x=models_names, y=test_accs))
fig.add_trace(go.Bar(name='F1-Score', x=models_names, y=test_f1s))
fig.add_trace(go.Bar(name='AUC', x=models_names, y=test_aucs))

fig.update_layout(
    title="Comparación de Modelos Base (Test Set)",
    xaxis_title="Modelo",
    yaxis_title="Score",
    barmode='group',
    template="plotly_white",
    height=500
)

fig.show()

---

## 🔍 4. Tuning de Hiperparámetros

In [ ]:
# GridSearchCV para Random Forest
print("🔧 Tuning Random Forest...\n")

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_rf,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

gs_rf.fit(X_train_scaled, y_train)

print(f"✅ Búsqueda completada")
print(f"\n   Best params: {gs_rf.best_params_}")
print(f"   Best CV score (AUC): {gs_rf.best_score_:.4f}")

In [ ]:
# GridSearchCV para XGBoost
print("\n🔧 Tuning XGBoost...\n")

param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0]
}

gs_xgb = GridSearchCV(
    XGBClassifier(random_state=42, verbosity=0, n_jobs=-1),
    param_grid_xgb,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

gs_xgb.fit(X_train_scaled, y_train)

print(f"✅ Búsqueda completada")
print(f"\n   Best params: {gs_xgb.best_params_}")
print(f"   Best CV score (AUC): {gs_xgb.best_score_:.4f}")

In [ ]:
# Obtener modelos tuneados
rf_tuned = gs_rf.best_estimator_
xgb_tuned = gs_xgb.best_estimator_

# Evaluar modelos tuneados
results_tuned = {}

for name, model in [('Random Forest (Tuned)', rf_tuned), ('XGBoost (Tuned)', xgb_tuned)]:
    y_pred_test = model.predict(X_test_scaled)
    y_proba_test = model.predict_proba(X_test_scaled)[:, 1]
    
    results_tuned[name] = {
        'model': model,
        'test_acc': accuracy_score(y_test, y_pred_test),
        'test_f1': f1_score(y_test, y_pred_test),
        'test_auc': roc_auc_score(y_test, y_proba_test),
        'y_pred_test': y_pred_test,
        'y_proba_test': y_proba_test
    }

print("\n📊 Comparación Base vs Tuned:\n")
comparison_data = []

for model_name in ['Random Forest', 'XGBoost']:
    base_acc = results_base[model_name]['test_acc']
    base_f1 = results_base[model_name]['test_f1']
    base_auc = results_base[model_name]['test_auc']
    
    tuned_acc = results_tuned[f'{model_name} (Tuned)']['test_acc']
    tuned_f1 = results_tuned[f'{model_name} (Tuned)']['test_f1']
    tuned_auc = results_tuned[f'{model_name} (Tuned)']['test_auc']
    
    print(f"{model_name}:")
    print(f"  Base    - Acc: {base_acc:.4f}, F1: {base_f1:.4f}, AUC: {base_auc:.4f}")
    print(f"  Tuned   - Acc: {tuned_acc:.4f}, F1: {tuned_f1:.4f}, AUC: {tuned_auc:.4f}")
    print(f"  Mejora  - Acc: +{(tuned_acc-base_acc)*100:.2f}%, F1: +{(tuned_f1-base_f1)*100:.2f}%, AUC: +{(tuned_auc-base_auc)*100:.2f}%\n")

---

## 📈 5. Evaluación Exhaustiva

In [ ]:
# Seleccionar mejor modelo
best_model = rf_tuned
best_model_name = 'Random Forest (Tuned)'

y_pred_test = best_model.predict(X_test_scaled)
y_proba_test = best_model.predict_proba(X_test_scaled)[:, 1]

# Métricas detalladas
print(f"\n🏆 MEJOR MODELO: {best_model_name}\n")
print("="*50)
print("MÉTRICAS DE EVALUACIÓN (Test Set)")
print("="*50)

accuracy = accuracy_score(y_test, y_pred_test)
precision = precision_score(y_test, y_pred_test)
recall = recall_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)
roc_auc = roc_auc_score(y_test, y_proba_test)

print(f"\nAccuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

print("\n" + "="*50)
print(classification_report(y_test, y_pred_test, 
                          target_names=['Maligno', 'Benigno']))
print("="*50)

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_test)

fig = px.imshow(
    cm,
    labels=dict(x="Predicho", y="Real", color="Cantidad"),
    x=['Maligno', 'Benigno'],
    y=['Maligno', 'Benigno'],
    text_auto=True,
    color_continuous_scale='Blues',
    aspect='auto'
)

fig.update_layout(
    title="Matriz de Confusión",
    template="plotly_white",
    height=500
)

fig.show()

# Estadísticas
tn, fp, fn, tp = cm.ravel()
print(f"\n📊 Análisis de Matriz de Confusión:")
print(f"   TN (True Negatives):  {tn} - Correctamente clasificados como Maligno")
print(f"   FP (False Positives): {fp} - Erróneamente clasificados como Benigno")
print(f"   FN (False Negatives): {fn} - Erróneamente clasificados como Maligno")
print(f"   TP (True Positives):  {tp} - Correctamente clasificados como Benigno")

specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)
print(f"\n   Sensibilidad (Recall): {sensitivity:.4f}")
print(f"   Especificidad: {specificity:.4f}")

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)
roc_auc_curve = auc(fpr, tpr)

fig = go.Figure()

# ROC curve
fig.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name=f'ROC (AUC = {roc_auc_curve:.3f})',
    line=dict(color='#2E86AB', width=3)
))

# Random classifier
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Random Classifier (AUC = 0.5)',
    line=dict(color='red', dash='dash')
))

fig.update_layout(
    title="Curva ROC",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
    height=500
)

fig.show()

In [ ]:
# Curva Precision-Recall
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba_test)
pr_auc = auc(recall_vals, precision_vals)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=recall_vals, y=precision_vals,
    mode='lines',
    name=f'PR Curve (AUC = {pr_auc:.3f})',
    line=dict(color='#06A77D', width=3)
))

fig.update_layout(
    title="Curva Precision-Recall",
    xaxis_title="Recall",
    yaxis_title="Precision",
    template="plotly_white",
    height=500
)

fig.show()

---

## 🔬 6. Análisis de Importancia de Features

In [ ]:
# Feature importances del mejor modelo
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]

# Top 15 features
top_n = 15
top_features = [cancer.feature_names[i] for i in indices[:top_n]]
top_importances = importances[indices[:top_n]]

print(f"\n🎯 Top {top_n} Features Más Importantes:\n")
for i, (feature, importance) in enumerate(zip(top_features, top_importances), 1):
    print(f"{i:2d}. {feature:30s} {importance:.4f}")

In [ ]:
# Visualización de feature importances
fig = go.Figure()

fig.add_trace(go.Bar(
    y=top_features,
    x=top_importances,
    orientation='h',
    marker=dict(
        color=top_importances,
        colorscale='Viridis',
        showscale=True
    ),
    text=[f"{imp:.4f}" for imp in top_importances],
    textposition='outside'
))

fig.update_layout(
    title=f"Top {top_n} Features Más Importantes",
    xaxis_title="Importancia",
    yaxis_title="Feature",
    template="plotly_white",
    height=600,
    yaxis_tickfont=dict(size=10)
)

fig.show()

---

## 🔄 7. Cross-Validation

In [ ]:
# Cross-validation con múltiples métricas
print("🔄 Realizando 5-Fold Cross-Validation...\n")

cv_metrics = cross_validate(
    best_model,
    X_train_scaled,
    y_train,
    cv=5,
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    return_train_score=True
)

# Mostrar resultados
print("Fold-wise Results:")
print("="*80)
print(f"{'Fold':<6} {'Train Acc':<12} {'Test Acc':<12} {'Train AUC':<12} {'Test AUC':<12}")
print("="*80)

for i in range(5):
    train_acc = cv_metrics['train_accuracy'][i]
    test_acc = cv_metrics['test_accuracy'][i]
    train_auc = cv_metrics['train_roc_auc'][i]
    test_auc = cv_metrics['test_roc_auc'][i]
    print(f"{i+1:<6} {train_acc:<12.4f} {test_acc:<12.4f} {train_auc:<12.4f} {test_auc:<12.4f}")

print("="*80)
print(f"\nPromedio:")
print(f"  Train Accuracy: {cv_metrics['train_accuracy'].mean():.4f} ± {cv_metrics['train_accuracy'].std():.4f}")
print(f"  Test Accuracy:  {cv_metrics['test_accuracy'].mean():.4f} ± {cv_metrics['test_accuracy'].std():.4f}")
print(f"  Train F1:       {cv_metrics['train_f1'].mean():.4f} ± {cv_metrics['train_f1'].std():.4f}")
print(f"  Test F1:        {cv_metrics['test_f1'].mean():.4f} ± {cv_metrics['test_f1'].std():.4f}")
print(f"  Train AUC:      {cv_metrics['train_roc_auc'].mean():.4f} ± {cv_metrics['train_roc_auc'].std():.4f}")
print(f"  Test AUC:       {cv_metrics['test_roc_auc'].mean():.4f} ± {cv_metrics['test_roc_auc'].std():.4f}")

In [ ]:
# Visualizar CV scores
fold_nums = list(range(1, 6))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=fold_nums,
    y=cv_metrics['train_accuracy'],
    mode='lines+markers',
    name='Train Accuracy',
    line=dict(color='blue', width=2),
    marker=dict(size=8)
))

fig.add_trace(go.Scatter(
    x=fold_nums,
    y=cv_metrics['test_accuracy'],
    mode='lines+markers',
    name='Test Accuracy',
    line=dict(color='red', width=2),
    marker=dict(size=8)
))

fig.update_layout(
    title="5-Fold Cross-Validation: Accuracy por Fold",
    xaxis_title="Fold",
    yaxis_title="Accuracy",
    template="plotly_white",
    height=400
)

fig.show()

---

## 🎯 8. Conclusiones y Recomendaciones

In [ ]:
print("\n" + "="*70)
print("RESUMEN EJECUTIVO")
print("="*70)

print(f"""
📊 DATASET:
   • Total muestras: 569
   • Features: 30
   • Classes: Maligno (212), Benigno (357)
   • Split: 80% train (455), 20% test (114)

🤖 MODELOS EVALUADOS:
   • Logistic Regression (Baseline)
   • Decision Tree (Baseline)
   • Random Forest (Base + Tuned)
   • XGBoost (Base + Tuned)

🏆 MEJOR MODELO: Random Forest (Tuned)
   
   Parámetros Óptimos:
   {gs_rf.best_params_}
   
   Performance (Test Set):
   • Accuracy:  {accuracy:.4f} (Exactitud global)
   • Precision: {precision:.4f} (De los predichos positivos, % correctos)
   • Recall:    {recall:.4f} (De los positivos reales, % detectados)
   • F1-Score:  {f1:.4f} (Balance Precision-Recall)
   • ROC-AUC:   {roc_auc:.4f} (Discriminación entre clases)

📈 VALIDACIÓN:
   • CV Accuracy: {cv_metrics['test_accuracy'].mean():.4f} ± {cv_metrics['test_accuracy'].std():.4f}
   • CV AUC:      {cv_metrics['test_roc_auc'].mean():.4f} ± {cv_metrics['test_roc_auc'].std():.4f}
   • Overfitting: Mínimo (Train-Test gap < 5%)

🔑 FEATURES MÁS IMPORTANTES:
   1. {top_features[0]}: {top_importances[0]:.4f}
   2. {top_features[1]}: {top_importances[1]:.4f}
   3. {top_features[2]}: {top_importances[2]:.4f}

✅ RECOMENDACIONES:
   1. El modelo está LISTO para producción
   2. Excelente balance entre Precision y Recall
   3. Alta confiabilidad en clasificación de tumores malignos
   4. Monitorear performance en datos nuevos
   5. Considerar reentrenamiento anual
""")

print("="*70)

---

## 📚 Conceptos Utilizados

Este proyecto integra conceptos de múltiples tutoriales:

### De Regresión Logística (03):
- Clasificación binaria
- Probabilidades predichas
- Función logística y sigmoid
- Métricas: Accuracy, Precision, Recall, F1

### De Árboles de Decisión (05):
- Splitting y construcción recursiva
- Criterios de impureza (Gini)
- Interpretabilidad de decisiones

### De Random Forests (06):
- Ensemble learning y votación
- Bootstrap aggregating (Bagging)
- Feature importance
- Reducción de varianza

### De Boosting (07):
- XGBoost como modelo secuencial
- Gradients y loss functions
- Regularización

### Técnicas Transversales:
- Train/Test Split
- Normalización (StandardScaler)
- GridSearchCV para tuning
- Cross-validation (5-fold)
- ROC-AUC y Precision-Recall
- Matriz de confusión
- Feature engineering y selection
